# 🔗 Clase 4 — Joins: combinando fuentes de datos
**Maestría en Fintech — Programación para el Análisis de Datos**

**Pregunta que responde:** *Mis datos de clientes están en una tabla y las transacciones en otra. ¿Cómo las uno?*

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
print('✅ Listo')

In [ ]:
# Los datos se descargan solos desde el repo del curso.
# No hace falta subir ningún archivo a Colab.
DATOS = 'https://raw.githubusercontent.com/camilojaure/itba-pad/main/datasets/'

# Cargamos las tres tablas
clientes      = pd.read_csv(DATOS + 'clientes.csv')
tarjetas      = pd.read_csv(DATOS + 'tarjetas.csv')
transacciones = pd.read_csv(DATOS + 'transacciones.csv')

print(f'clientes:      {len(clientes):,} filas')
print(f'tarjetas:      {len(tarjetas):,} filas')
print(f'transacciones: {len(transacciones):,} filas')


## 1. ¿Por qué existen múltiples tablas?

En cualquier empresa los datos viven separados: una tabla de clientes, otra de productos, otra de transacciones. Los joins son la herramienta para unirlos sin duplicar información.

```
clientes      →  ¿quién es el cliente?
tarjetas      →  ¿qué tarjeta tiene?
transacciones →  ¿qué compró?
```

## 2. Tipos de join

```
INNER JOIN  →  Solo los que están en AMBAS tablas
LEFT JOIN   →  Todos los de la izquierda, con lo que matchea de la derecha
RIGHT JOIN  →  Todos los de la derecha, con lo que matchea de la izquierda
OUTER JOIN  →  Todos de ambas tablas
```

> 💡 El 80% de las veces vas a usar LEFT JOIN — querés mantener todos tus clientes aunque no tengan tarjeta o transacciones.

In [ ]:
# INNER JOIN: clientes que tienen al menos una tarjeta
clientes_con_tarjeta = pd.merge(clientes, tarjetas, on='cliente_id', how='inner')
print(f'Clientes con al menos una tarjeta: {clientes_con_tarjeta["cliente_id"].nunique():,}')
clientes_con_tarjeta[['cliente_id','nombre','segmento','tipo_tarjeta','limite_credito']].head()

In [ ]:
# LEFT JOIN: todos los clientes, con tarjetas si tienen
todos = pd.merge(clientes, tarjetas, on='cliente_id', how='left')
sin_tarjeta = todos[todos['tarjeta_id'].isna()]
print(f'Clientes SIN tarjeta registrada: {sin_tarjeta["cliente_id"].nunique()}')
print(f'Total clientes: {todos["cliente_id"].nunique():,}')

In [ ]:
# Ahora sumamos las transacciones — join en cadena
# Paso 1: clientes + tarjetas
base = pd.merge(clientes[['cliente_id','segmento','provincia','edad','churn']],
                tarjetas[['tarjeta_id','cliente_id','tipo_tarjeta','limite_credito']],
                on='cliente_id', how='left')

# Paso 2: + transacciones
df = pd.merge(base,
              transacciones[['transaccion_id','tarjeta_id','fecha','monto_ars','categoria','aprobada']],
              on='tarjeta_id', how='left')

print(f'Tabla analítica completa: {len(df):,} filas, {df.shape[1]} columnas')
df.head()

## 3. Analizando la tabla combinada

Ahora que tenemos todo junto, podemos responder preguntas que antes requerían múltiples pasos.

In [ ]:
# ¿Cuánto gastó en promedio cada segmento?
df.groupby('segmento')['monto_ars'].agg(['mean','sum','count']).round(0)

In [ ]:
# ¿Los clientes con churn gastaban menos antes de irse?
comparacion = df.groupby('churn')['monto_ars'].agg(['mean','median']).round(0)
comparacion.index = ['Sin churn','Con churn']
comparacion

In [ ]:
# ¿Qué categorías usan más los clientes Premium vs Retail?
pivot = df[df['segmento'].isin(['Premium','Retail'])].pivot_table(
    values='monto_ars', index='categoria', columns='segmento', aggfunc='mean').round(0)
pivot.sort_values('Premium', ascending=False)

In [ ]:
# Ranking de clientes por gasto total
ranking = df.groupby(['cliente_id','segmento'])['monto_ars'].sum().reset_index()
ranking = ranking.sort_values('monto_ars', ascending=False)
ranking['monto_ars'] = ranking['monto_ars'].apply(lambda x: f'${x:,.0f}')
ranking.head(10)

## 4. concat — Apilando datasets

Cuando tenés datos del mismo tipo en archivos separados (ej: transacciones de enero, febrero, marzo)

In [ ]:
# Simulamos 3 meses de transacciones separadas
trans_q1 = transacciones[transacciones['fecha'].str.startswith('2024-0')]
trans_q2 = transacciones[transacciones['fecha'].str[5:7].isin(['04','05','06'])]

print(f'Q1: {len(trans_q1):,} transacciones')
print(f'Q2: {len(trans_q2):,} transacciones')

# Las apilamos verticalmente
combinado = pd.concat([trans_q1, trans_q2], ignore_index=True)
print(f'Combinado: {len(combinado):,} transacciones')

---
## 🧑‍💻 Práctica — Tu turno

In [ ]:
# EJERCICIO 1
# ¿Cuántas tarjetas de crédito tiene cada segmento en promedio?
# Tip: merge clientes + tarjetas, filtrar tipo que contenga 'Crédito', groupby segmento
# Tu código acá:


In [ ]:
# EJERCICIO 2
# ¿Cuál es el límite de crédito promedio por segmento?
# Tu código acá:


In [ ]:
# EJERCICIO 3
# ¿Qué provincia genera más volumen de transacciones?
# Usá la tabla combinada (df) y agrupá por provincia.
# Tu código acá:


In [ ]:
# EJERCICIO 4
# Encontrá los clientes que tienen tarjeta pero NUNCA hicieron una transacción.
# Tip: merge tarjetas + transacciones con how='left', filtrá nulos en transaccion_id
# Tu código acá:


In [ ]:
# EJERCICIO 5
# ¿Hay diferencia en el tipo de categorías que consumen clientes con vs sin churn?
# Mostralo en un pivot o gráfico comparativo.
# Tu código acá:


In [ ]:
# EJERCICIO 6
# Calculá el 'share of wallet' por segmento: qué % del gasto total corresponde a cada uno.
# Tu código acá:


In [ ]:
# EJERCICIO 7
# Creá un merge entre clientes y un resumen de sus transacciones (monto total, cantidad, ticket promedio).
# El resultado debe tener una fila por cliente.
# Tu código acá:


In [ ]:
# EJERCICIO 8
# ¿Cuál es la tasa de transacciones rechazadas por segmento?
# Tu código acá:


In [ ]:
# EJERCICIO 9
# Construí un reporte mensual por segmento: monto total de transacciones por mes y segmento.
# Graficalo como líneas separadas por segmento.
# Tu código acá:


In [ ]:
# EJERCICIO 10 — Desafío
# Armá una tabla de 'clientes valiosos en riesgo':
# clientes con churn == 1, gasto total > percentil 75 del total
# ¿Cuántos son? ¿Qué segmento dominan? ¿Cuánto representa su gasto sobre el total?
# Tu código acá:

# Conclusión:
# 
